# GraphBench: Graph-Based RAG Benchmark

End-to-end notebook for all phases: data ingestion, GNN training, benchmark, and results.

**Runtime**: Colab Pro (GPU, High-RAM)  
**Checkpoints**: Google Drive at `/content/drive/MyDrive/graphbench/`

| Phase | Description |
|-------|-------------|
| 1 | Setup & Dependencies |
| 2 | Data Pipeline: REBEL Triples → FAISS + Neo4j |
| 3 | GNN Training: 3-Layer GAT |
| 4 | Pipelines: GraphRAG & GNN-RAG |
| 5 | Benchmark: 500 HotpotQA Questions |
| 6 | Results & Analysis |
| 7 | Export for Demo |

## Phase 1 — Setup & Dependencies

In [ ]:
# Install graphbench-kg from PyPI (or from source for development)
!pip install graphbench-kg -q
# Alternatively, install from GitHub main:
# !pip install git+https://github.com/Rohanjain2312/graphbench.git -q

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive

drive.mount("/content/drive")

import os
from pathlib import Path

DRIVE_BASE = Path("/content/drive/MyDrive/graphbench")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Sub-directories
CHECKPOINT_DIR = DRIVE_BASE / "checkpoints"
RESULTS_DIR = DRIVE_BASE / "results"
FAISS_PATH = DRIVE_BASE / "faiss_index"
for d in [CHECKPOINT_DIR, RESULTS_DIR]:
    d.mkdir(exist_ok=True)

print("Drive mounted. Base dir:", DRIVE_BASE)

In [ ]:
# Load environment variables from Drive .env file
# Copy your .env to /content/drive/MyDrive/graphbench/.env before running
from dotenv import load_dotenv

env_path = DRIVE_BASE / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print(f".env loaded from {env_path}")
else:
    print(f"WARNING: No .env found at {env_path}")
    print(
        "Set NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, HF_TOKEN, WANDB_API_KEY manually:"
    )
    # Uncomment and fill in if no .env file:
    # import os
    # os.environ['NEO4J_URI']      = 'neo4j+s://YOUR_AURADB_URI'
    # os.environ['NEO4J_USERNAME'] = 'neo4j'
    # os.environ['NEO4J_PASSWORD'] = 'YOUR_PASSWORD'
    # os.environ['HF_TOKEN']       = 'hf_YOUR_TOKEN'
    # os.environ['WANDB_API_KEY']  = 'YOUR_WANDB_KEY'

In [ ]:
# Verify installation and GPU
from graphbench import __version__
import torch

print(f"graphbench-kg version : {__version__}")
print(f"PyTorch version       : {torch.__version__}")
print(f"CUDA available        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                   : {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM                  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )

## Phase 2 — Data Pipeline: REBEL Triples → FAISS + Neo4j

In [ ]:
# Initialise Neo4j client and verify connectivity
from graphbench.utils.neo4j_client import Neo4jClient

neo4j = Neo4jClient()
neo4j.ensure_schema()
print(f"Neo4j connected. Existing triples: {neo4j.count_triples():,}")

In [ ]:
# Extract REBEL triples from HotpotQA passages
# This runs REBEL inference on GPU — ~2-3 hours for 50k triples on Colab T4
# Skip this cell if you have a pre-extracted .parquet file on Drive
from graphbench.ingestion.rebel_loader import stream_triples
from tqdm import tqdm

MAX_TRIPLES = 50_000
print(f"Streaming up to {MAX_TRIPLES:,} REBEL triples from HotpotQA...")

triples = []
for triple in tqdm(
    stream_triples(max_triples=MAX_TRIPLES), total=MAX_TRIPLES, desc="Extracting"
):
    triples.append(triple)

print(f"\nExtracted {len(triples):,} triples")
print("Sample:", triples[:3])

In [ ]:
# Write triples to Neo4j AuraDB
from graphbench.ingestion.neo4j_writer import write_triples

print("Writing triples to Neo4j...")
write_triples(neo4j, triples, batch_size=500)
print(f"Neo4j triple count: {neo4j.count_triples():,}")

In [ ]:
# Embed all unique entities with all-MiniLM-L6-v2
from graphbench.ingestion.embedder import embed_entities

entities = sorted({t["subject"] for t in triples} | {t["object"] for t in triples})
print(f"Embedding {len(entities):,} unique entities...")

entity_strings, embeddings = embed_entities(entities)
print(f"Embedding matrix: {embeddings.shape}  dtype={embeddings.dtype}")

In [ ]:
# Build and save FAISS index to Drive
from graphbench.ingestion.faiss_writer import build_and_save_index

index = build_and_save_index(entity_strings, embeddings, index_path=FAISS_PATH)
print(f"FAISS index: {index.ntotal:,} vectors saved to {FAISS_PATH}.faiss")

# Build embedding dict for GNN training
embedding_dict = dict(zip(entity_strings, embeddings))
print(f"embedding_dict: {len(embedding_dict):,} entries")

In [ ]:
# Quick sanity check
from graphbench.utils.faiss_client import FAISSClient
import numpy as np

faiss_client = FAISSClient.load(FAISS_PATH)
print(f"FAISSClient loaded: {faiss_client.size:,} vectors")

# Test search
test_vec = embeddings[0:1]  # first entity
results = faiss_client.search(test_vec[0], k=5)
print(f'Top-5 neighbours of "{entity_strings[0]}": {[r[0] for r in results]}')

## Phase 3 — GNN Training: 3-Layer GAT

In [ ]:
# Build KGDataset with 80/10/10 train/val/test split
from graphbench.gnn.dataset import KGDataset

print("Building KGDataset...")
ds = KGDataset(triples, embedding_dict)
train_data, val_data, test_data = ds.split()

print(f"Nodes       : {train_data.x.shape[0]:,}")
print(
    f"Train edges : {train_data.edge_label_index.shape[1]:,}  "
    f"(pos={int(train_data.edge_label.sum()):,}, "
    f"neg={(train_data.edge_label == 0).sum().item():,})"
)
print(f"Val edges   : {val_data.edge_label_index.shape[1]:,}")
print(f"Test edges  : {test_data.edge_label_index.shape[1]:,}")

In [ ]:
# Train 3-layer GAT — target: test AUC-ROC > 0.75
from graphbench.gnn.model import GATModel
from graphbench.gnn.trainer import train_gnn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

model = GATModel()
results = train_gnn(
    model=model,
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    checkpoint_dir=CHECKPOINT_DIR,
    device=device,
    max_epochs=200,
    lr=1e-3,
    early_stopping_patience=15,
)

print(f"\nTest AUC-ROC   : {results['test_auc']:.4f}")
print(f"Test Loss      : {results['test_loss']:.4f}")
print(
    f"Best Val AUC   : {results['best_val_auc']:.4f}  (epoch {results['best_epoch']})"
)

if results["test_auc"] < 0.75:
    print("\n⚠️  AUC below 0.75 threshold — consider more epochs or lower LR.")
else:
    print("\n✅ AUC threshold met — proceed to Phase 4.")

In [ ]:
# Load best checkpoint and verify weights round-trip
from graphbench.utils.checkpoint import load_best_checkpoint

checkpoint, ckpt_path = load_best_checkpoint(CHECKPOINT_DIR)
print(f"Best checkpoint : {ckpt_path.name}")
print(f'Epoch           : {checkpoint["epoch"]}')
print(f'Val AUC         : {checkpoint["val_auc"]:.4f}')

# Restore model for pipeline use
gat_model = GATModel()
gat_model.load_state_dict(checkpoint["model_state_dict"])
gat_model.eval()
print("Model restored from checkpoint.")

## Phase 4 — Pipelines: GraphRAG & GNN-RAG

In [ ]:
# Initialise both pipelines
from graphbench.utils.llm_client import LLMClient
from graphbench.community.detector import CommunityDetector
from graphbench.pipelines.graphrag_pipeline import GraphRAGPipeline
from graphbench.pipelines.gnnrag_pipeline import GNNRAGPipeline

# LLM: Mistral-7B-Instruct-v0.2 in 4-bit on GPU
llm = LLMClient(backend="hf")

graphrag = GraphRAGPipeline(
    neo4j_client=neo4j,
    faiss_client=faiss_client,
    llm_client=llm,
)

gnnrag = GNNRAGPipeline(
    neo4j_client=neo4j,
    faiss_client=faiss_client,
    llm_client=llm,
    gat_model=gat_model,
    entity_embeddings=embedding_dict,
)

print("GraphRAG  :", graphrag.name)
print("GNN-RAG   :", gnnrag.name)

In [ ]:
# Smoke test on 5 questions
import time
import pandas as pd

smoke_questions = [
    "Where was Marie Curie born?",
    "What did Albert Einstein win the Nobel Prize for?",
    "Which country is Warsaw located in?",
    "Who discovered penicillin?",
    "In what field did Marie Curie work?",
]

rows = []
for q in smoke_questions:
    t0 = time.perf_counter()
    ra = graphrag.answer(q)
    lat_a = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    rb = gnnrag.answer(q)
    lat_b = (time.perf_counter() - t0) * 1000

    rows.append(
        {
            "Question": q,
            "GraphRAG Answer": ra.predicted_answer,
            "GraphRAG ms": f"{lat_a:.0f}",
            "GNN-RAG Answer": rb.predicted_answer,
            "GNN-RAG ms": f"{lat_b:.0f}",
        }
    )

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## Phase 5 — Benchmark: 500 HotpotQA Questions

In [ ]:
# Run full benchmark — ~2-4 hours depending on GPU and LLM latency
from graphbench.benchmark.evaluator import Evaluator

evaluator = Evaluator(
    pipeline_a=graphrag,
    pipeline_b=gnnrag,
    n_questions=500,
    seed=42,
    results_dir=RESULTS_DIR,
)

summary = evaluator.run()
print("\nBenchmark complete.")

In [ ]:
# Summary table
import pandas as pd

rows = []
for pipeline_name, metrics in summary.items():
    rows.append(
        {
            "Pipeline": pipeline_name,
            "EM (%)": f"{metrics['em'] * 100:.2f}",
            "F1 (%)": f"{metrics['f1'] * 100:.2f}",
            "Latency P50 (ms)": f"{metrics['latency_p50']:.0f}",
            "Latency P95 (ms)": f"{metrics['latency_p95']:.0f}",
            "N Questions": metrics["n_questions"],
        }
    )

df_summary = pd.DataFrame(rows).set_index("Pipeline")
print("=== Benchmark Results ===")
print(df_summary.to_string())

## Phase 6 — Results & Analysis

In [ ]:
# Load the most recent results file
import json, glob
from pathlib import Path

result_files = sorted(Path(RESULTS_DIR).glob("*_results.json"))
assert result_files, f"No results found in {RESULTS_DIR}. Run Phase 5 first."

with open(result_files[-1]) as f:
    results = json.load(f)

summary = results["summary"]
rows_data = results["rows"]
print(f"Loaded {len(rows_data)} question results from {result_files[-1].name}")

In [ ]:
# EM & F1 comparison bar chart
import plotly.graph_objects as go

pipelines = list(summary.keys())
em_vals = [summary[p]["em"] * 100 for p in pipelines]
f1_vals = [summary[p]["f1"] * 100 for p in pipelines]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        name="Exact Match (%)",
        x=pipelines,
        y=em_vals,
        marker_color="#4e79a7",
        text=[f"{v:.1f}%" for v in em_vals],
        textposition="outside",
    )
)
fig.add_trace(
    go.Bar(
        name="Token F1 (%)",
        x=pipelines,
        y=f1_vals,
        marker_color="#f28e2b",
        text=[f"{v:.1f}%" for v in f1_vals],
        textposition="outside",
    )
)
fig.update_layout(
    title="GraphRAG vs GNN-RAG: Exact Match & F1",
    yaxis_title="Score (%)",
    yaxis_range=[0, 105],
    barmode="group",
    template="plotly_white",
)
fig.show()

In [ ]:
# Latency comparison
p50_vals = [summary[p]["latency_p50"] for p in pipelines]
p95_vals = [summary[p]["latency_p95"] for p in pipelines]

fig2 = go.Figure()
fig2.add_trace(
    go.Bar(
        name="P50 (ms)",
        x=pipelines,
        y=p50_vals,
        marker_color="#59a14f",
        text=[f"{v:.0f} ms" for v in p50_vals],
        textposition="outside",
    )
)
fig2.add_trace(
    go.Bar(
        name="P95 (ms)",
        x=pipelines,
        y=p95_vals,
        marker_color="#e15759",
        text=[f"{v:.0f} ms" for v in p95_vals],
        textposition="outside",
    )
)
fig2.update_layout(
    title="Latency Comparison (ms)",
    yaxis_title="Latency (ms)",
    barmode="group",
    template="plotly_white",
)
fig2.show()

In [ ]:
# Break down EM/F1 by question type (bridge vs comparison)
import pandas as pd
import numpy as np
from graphbench.benchmark.metrics import exact_match, token_f1

df_rows = pd.DataFrame(rows_data)

for pipeline_col_prefix in [p.replace("-", "_") for p in pipelines]:
    original_name = (
        pipeline_col_prefix.replace("_", "-")
        if "-" not in pipeline_col_prefix
        else pipeline_col_prefix
    )
    # find the matching column prefix

for pipeline in pipelines:
    pred_col = f"{pipeline}_predicted"
    if pred_col not in df_rows.columns:
        continue
    df_rows[f"{pipeline}_em"] = df_rows.apply(
        lambda r, p=pipeline: exact_match(r[f"{p}_predicted"], r["gold_answer"]), axis=1
    )
    df_rows[f"{pipeline}_f1"] = df_rows.apply(
        lambda r, p=pipeline: token_f1(r[f"{p}_predicted"], r["gold_answer"]), axis=1
    )

type_breakdown = []
for qtype in ["bridge", "comparison"]:
    subset = df_rows[df_rows["type"] == qtype]
    row = {"type": qtype, "n": len(subset)}
    for pipeline in pipelines:
        em_col = f"{pipeline}_em"
        f1_col = f"{pipeline}_f1"
        if em_col in subset.columns:
            row[f"{pipeline} EM"] = f"{subset[em_col].mean() * 100:.1f}%"
            row[f"{pipeline} F1"] = f"{subset[f1_col].mean() * 100:.1f}%"
    type_breakdown.append(row)

print("=== Results by Question Type ===")
print(pd.DataFrame(type_breakdown).to_string(index=False))

In [ ]:
# Show 5 most interesting error cases (high F1 for one, low for other)
if len(pipelines) >= 2:
    pa, pb = pipelines[0], pipelines[1]
    em_a_col = f"{pa}_em"
    em_b_col = f"{pb}_em"
    if em_a_col in df_rows.columns and em_b_col in df_rows.columns:
        # Cases where GNN-RAG wins but GraphRAG fails
        gnn_wins = df_rows[(df_rows[em_b_col] == 1) & (df_rows[em_a_col] == 0)]
        print(f"Questions where {pb} wins ({len(gnn_wins)} total):")
        for _, r in gnn_wins.head(3).iterrows():
            print(f'  Q: {r["question"][:80]}')
            print(
                f'     Gold: {r["gold_answer"]}  |  {pa}: {r[f"{pa}_predicted"]}  |  {pb}: {r[f"{pb}_predicted"]}'
            )

In [ ]:
# Write results to docs/benchmark_results.md
import textwrap
from datetime import datetime, timezone

md_lines = [
    "# GraphBench Benchmark Results",
    "",
    f'*Generated: {datetime.now(timezone.utc).strftime("%Y-%m-%d")}*',
    "",
    "## Setup",
    "- Dataset: HotpotQA distractor (500 questions: 250 bridge + 250 comparison)",
    "- KG: ~50k REBEL triples in Neo4j AuraDB Free",
    "- Embeddings: sentence-transformers/all-MiniLM-L6-v2 (384-dim)",
    "- LLM: mistralai/Mistral-7B-Instruct-v0.2 (4-bit on T4)",
    "- GNN: 3-layer GAT (4 heads), trained to AUC-ROC > 0.75",
    "",
    "## Results",
    "",
    "| Pipeline | EM (%) | F1 (%) | Latency P50 (ms) | Latency P95 (ms) |",
    "|----------|--------|--------|-----------------|-----------------|",
]

for pipeline, metrics in summary.items():
    md_lines.append(
        f'| {pipeline} | {metrics["em"]*100:.2f} | {metrics["f1"]*100:.2f} '
        f'| {metrics["latency_p50"]:.0f} | {metrics["latency_p95"]:.0f} |'
    )

md_lines += [
    "",
    "## Notes",
    "- Latency measured with `time.perf_counter()` around `pipeline.answer()` only.",
    "- EM and F1 computed with `normalize_answer()` (lowercase, remove articles/punctuation).",
]

md_content = "\n".join(md_lines)

# Save to docs/ in repo (relative path for Colab)
import subprocess

result = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
)
repo_root = Path(result.stdout.strip()) if result.returncode == 0 else Path(".")
md_path = repo_root / "docs" / "benchmark_results.md"
md_path.parent.mkdir(exist_ok=True)
md_path.write_text(md_content)
print(f"benchmark_results.md written to {md_path}")
print()
print(md_content)

## Phase 7 — Export for Demo

In [ ]:
# Save summary.json for HuggingFace Spaces leaderboard tab
import json

summary_path = RESULTS_DIR / "summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"summary.json → {summary_path}")

In [ ]:
# List all assets needed for the HuggingFace Spaces demo
print("=== Assets to upload to HuggingFace Spaces ===")
assets = [
    (FAISS_PATH.parent / (FAISS_PATH.stem + ".faiss"), "FAISS index"),
    (FAISS_PATH.parent / (FAISS_PATH.stem + "_id_map.json"), "FAISS id_map"),
    (summary_path, "Benchmark summary"),
]
try:
    ckpt_path_demo = sorted(CHECKPOINT_DIR.glob("*.pt"))[-1]
    assets.append((ckpt_path_demo, "Best GAT checkpoint"))
except IndexError:
    pass

for path, label in assets:
    size_mb = path.stat().st_size / 1e6 if path.exists() else 0
    status = "✅" if path.exists() else "❌ MISSING"
    print(f"  {status}  {label:30s}  {size_mb:.1f} MB   {path}")

In [ ]:
# (Optional) Push assets to HuggingFace Hub as a dataset
# Uncomment and fill in your Space ID
#
# from huggingface_hub import HfApi
# api = HfApi()
# SPACE_ID = 'rohanjain2312/graphbench'  # <-- your HF Space
#
# for path, label in assets:
#     if path.exists():
#         api.upload_file(
#             path_or_fileobj=str(path),
#             path_in_repo=f'data/{path.name}',
#             repo_id=SPACE_ID,
#             repo_type='space',
#         )
#         print(f'Uploaded: {path.name}')
print("Uncomment the cell above to push assets to HuggingFace Spaces.")

In [ ]:
print("🎉 All phases complete!")
print()
for pipeline, metrics in summary.items():
    print(
        f'{pipeline:10s}  EM={metrics["em"]*100:.2f}%  F1={metrics["f1"]*100:.2f}%'
        f'  P50={metrics["latency_p50"]:.0f}ms  P95={metrics["latency_p95"]:.0f}ms'
    )